# Évaluation de la prévision du risque

Nous comparons un modèle Ridge explicable à une référence naïve : prévoir pour l'année suivante le même score que l'année actuelle. Le modèle n'est retenu que s'il réduit l'erreur sur 2024-2025.

## Protocole temporel

Chaque observation de l'année `t` sert à prévoir le score de `t+1`. Les paires jusqu'à la cible 2023-2024 servent à l'entraînement. La cible 2024-2025 reste entièrement réservée au test final. Aucun mélange aléatoire des années n'est réalisé.

In [1]:
from pathlib import Path
import sys
import pandas as pd

RACINE = Path.cwd()
if RACINE.name == '04_analyse':
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE / 'scripts'))
from evaluer_prevision import evaluer

rapport = evaluer(
    RACINE / '03_transformation/SCO-RISKBurkina_03-transformation_D1_engineer_indicateurs-province-annee_v1.csv',
    RACINE / '04_analyse/SCO-RISKBurkina_04-analyse_D1_data-scientist_previsions-2024-2025_v1.csv',
    RACINE / '04_analyse/SCO-RISKBurkina_04-analyse_D1_data-scientist_projection-2025-2026_v1.csv',
    RACINE / '04_analyse/SCO-RISKBurkina_04-analyse_D1_data-scientist_importance-modele_v1.csv',
    RACINE / '04_analyse/SCO-RISKBurkina_04-analyse_D1_data-scientist_modele-ridge_v1.joblib',
    RACINE / 'rapports_prives/07_evaluation_modele.json',
)
rapport

{'rapport': 'evaluation_prevision_risque',
 'cree_le_utc': '2026-11-16T10:33:39.356608+00:00',
 'protocole': "Paires année t vers t+1 ; entraînement jusqu'à 2023-2024 ; test cible 2024-2025.",
 'lignes_entrainement': 270,
 'lignes_test': 45,
 'variables': ['score_risque',
  'taux_abandon',
  'taux_redoublement',
  'taux_achevement',
  'delta_abandon',
  'delta_redoublement',
  'delta_score'],
 'selection_alpha': [{'alpha': 0.1,
   'mae_validation_temporelle': 9.909781329617228},
  {'alpha': 1.0, 'mae_validation_temporelle': 9.90055513519193},
  {'alpha': 10.0, 'mae_validation_temporelle': 9.833274332337158},
  {'alpha': 100.0, 'mae_validation_temporelle': 9.919098295289716}],
 'alpha_retenu': 10.0,
 'reference_naive': {'mae': 13.9653,
  'rmse': 18.6061,
  'correlation_rangs_spearman': 0.3037,
  'provinces_communes_top10': 3},
 'ridge': {'mae': 9.5488,
  'rmse': 12.4698,
  'correlation_rangs_spearman': 0.6388,
  'provinces_communes_top10': 6},
 'modele_retenu': 'ridge',
 'annee_projecti

In [2]:
comparaison = pd.DataFrame([
    {'méthode': 'Référence naïve', **rapport['reference_naive']},
    {'méthode': 'Ridge', **rapport['ridge']},
])
comparaison

,méthode,mae,rmse,correlation_rangs_spearman,provinces_communes_top10
0,Référence naïve,13.9653,18.6061,0.3037,3
1,Ridge,9.5488,12.4698,0.6388,6


In [3]:
previsions = pd.read_csv(RACINE / '04_analyse/SCO-RISKBurkina_04-analyse_D1_data-scientist_previsions-2024-2025_v1.csv')
projection_future = pd.read_csv(RACINE / '04_analyse/SCO-RISKBurkina_04-analyse_D1_data-scientist_projection-2025-2026_v1.csv')
importance = pd.read_csv(RACINE / '04_analyse/SCO-RISKBurkina_04-analyse_D1_data-scientist_importance-modele_v1.csv')
print('Modèle retenu :', rapport['modele_retenu'])
display(importance)
display(previsions.head(15))
projection_future.head(15)

Modèle retenu : ridge


,variable,coefficient_standardise,importance_absolue
0,score_risque,7.869465,7.869465
1,taux_achevement,-5.754097,5.754097
2,taux_abandon,-3.571281,3.571281
3,delta_redoublement,-1.458288,1.458288
4,delta_score,-1.404657,1.404657
5,taux_redoublement,0.804091,0.804091
6,delta_abandon,0.401882,0.401882


,province,region_2025,annee_source,annee_cible,score_observe,prediction_reference,prediction_ridge,prediction_retenue,modele_retenu
0,Oudalan,Liptako,2023-2024,2024-2025,77.56,83.11,76.29,76.29,ridge
1,Tapoa,Tapoa,2023-2024,2024-2025,76.22,83.33,72.85,72.85,ridge
2,Kossi,Sourou,2023-2024,2024-2025,45.78,86.78,72.03,72.03,ridge
3,Komondjari,Sirba,2023-2024,2024-2025,32.44,72.89,67.70,67.70,ridge
4,Kompienga,Goulmou,2023-2024,2024-2025,76.11,59.11,65.97,65.97,ridge
5,Bale,Bankui,2023-2024,2024-2025,64.44,68.44,65.51,65.51,ridge
6,Bam,Koulse,2023-2024,2024-2025,40.67,68.22,64.69,64.69,ridge
7,Yatenga,Yaadga,2023-2024,2024-2025,70.33,77.11,64.54,64.54,ridge
8,Seno,Liptako,2023-2024,2024-2025,72.44,47.44,64.54,64.54,ridge
9,Soum,Soum,2023-2024,2024-2025,69.11,42.89,62.92,62.92,ridge


,province,region_2025,annee_source,annee_cible,score_actuel,score_prevu,variation_prevue,modele
0,Oudalan,Liptako,2024-2025,2025-2026,77.56,79.27,1.71,ridge
1,Kompienga,Goulmou,2024-2025,2025-2026,76.11,76.30,0.19,ridge
2,Seno,Liptako,2024-2025,2025-2026,72.44,73.84,1.40,ridge
3,Tapoa,Tapoa,2024-2025,2025-2026,76.22,72.01,-4.21,ridge
4,Bale,Bankui,2024-2025,2025-2026,64.44,71.77,7.33,ridge
5,Gnagna,Sirba,2024-2025,2025-2026,68.22,70.25,2.03,ridge
6,Soum,Soum,2024-2025,2025-2026,69.11,68.72,-0.39,ridge
7,Nayala,Sourou,2024-2025,2025-2026,45.22,68.53,23.31,ridge
8,Gourma,Goulmou,2024-2025,2025-2026,60.22,67.70,7.48,ridge
9,Yagha,Liptako,2024-2025,2025-2026,60.67,67.08,6.41,ridge


## Interprétation prudente

La prévision sert à déclencher une revue humaine et à prioriser les provinces. Elle ne détermine pas automatiquement une allocation de ressources et ne produit aucune prédiction individuelle sur les élèves.